In [23]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.2f}'.format)

drive_file_path = '/content/drive/MyDrive/Jet2premium/web_traffic.csv'

df = pd.read_csv(drive_file_path)
print(f"Row: {len(df):,} / Cols: {df.shape[1]}")
print(df.head())

#chuyển data sang datatime
df['date'] = pd.to_datetime(df['date'])

print(f"Đọc file thành công")
print(f"   Rows : {df.shape[0]:,}")
print(f"   Cols : {df.shape[1]}")
print(f"\n--- 5 dòng đầu ---")
print(df.head(5).to_string())

#data structure
print("=" * 55)
print("Structure và Type")
print("=" * 55)
df.info()
print(df.dtypes)

#missing values check
print("=" * 55)
print("MISSING VALUES")
print("=" * 55)

null_counts = df.isnull().sum()
null_pct    = (null_counts / len(df) * 100).round(2)
miss_df     = pd.DataFrame({'null_count': null_counts, 'null_%': null_pct})
print(miss_df.to_string())

# Kiểm tra null ẩn dạng string
fake_nulls = ['', 'None', 'none', 'NULL', 'null', 'nan', 'N/A', 'NA', '-']
print("\nNull ẩn dạng string:")
found_fake = False
for col in df.select_dtypes(include='object').columns:
    n = df[col].isin(fake_nulls).sum()
    if n > 0:
        print(f"{col}: {n} giá trị nghi ngờ")
        found_fake = True
if not found_fake:
    print("Không phát hiện null ẩn")

#Duplicates check
print("=" * 55)
print("DUPLICATES")
print("=" * 55)

dup_rows = df.duplicated().sum()
print(f"Duplicate rows        : {str(dup_rows) if dup_rows else '0'}")

dup_pk = df.duplicated(subset=['date', 'traffic_source']).sum()
print(f"Duplicate (date+src)  : {str(dup_pk) if dup_pk else '0'}")

# Tổng số ngày có dữ liệu
print(f"\nSố ngày không bị trùng trong data : {df['date'].nunique():,}")

#descriptive statistics
print("=" * 55)
print("DESCRIPTIVE STATISTICS")
print("=" * 55)

num_cols = ['sessions', 'unique_visitors', 'page_views',
            'bounce_rate', 'avg_session_duration_sec']
print(df[num_cols].describe().round(4).to_string())

# Kiểm tra có giá trị âm hay không
print("\nKiểm tra giá trị âm:")
for col in num_cols:
    neg = (df[col] < 0).sum()
    status = f"{neg} giá trị âm" if neg else "Hợp lệ"
    print(f"   {col:<30}: {status}")

# Kiểm tra xem bounce_rate có nằm trong khoảng từ 0-1
out_of_range = ((df['bounce_rate'] < 0) | (df['bounce_rate'] > 1)).sum()
print(f"\nbounce_rate ngoài [0,1] : {str(out_of_range) if out_of_range else '0'}")

#phân bổ TS
print("=" * 55)
print("PHÂN BỐ TRAFFIC_SOURCE")
print("=" * 55)

src_counts = df['traffic_source'].value_counts()
print(f"\n{'Nguồn':<20} {'Số ngày':>8}  {'Tỷ lệ':>7}")
print("-" * 40)
for src, cnt in src_counts.items():
    pct = cnt / len(df) * 100
    print(f"{src:<20} {cnt:>8,}  {pct:>6.1f}%")

#tính bounce race trả lời câu 4
print("=" * 55)
print("TÍNH BOUNCE RACE CÂU 4")
print("=" * 55)

# Tính bounce_rate trung bình, median, min, max
bounce_stats = df.groupby('traffic_source')['bounce_rate'].agg(
    count  = 'count',       # số ngày xuất hiện
    mean   = 'mean',        # trung bình bounce_rate
    median = 'median',
    std    = 'std',
    min    = 'min',
    max    = 'max'
).sort_values('mean')       # sắp xếp

print("\nBảng bounce_rate sắp xếp tăng dần theo mean:")
print(bounce_stats.round(6).to_string())

# tỷ lệ bounce_rate trung bình min
best_source  = bounce_stats['mean'].idxmin()   # tên nguồn
best_value   = bounce_stats['mean'].min()       # giá trị

print(f"\n{'─'*50}")
print(f"  Nguồn có bounce_rate trung bình thấp nhất là:")
print(f"  → {best_source}  ({best_value:.6f})")
print(f"{'─'*50}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Row: 3,652 / Cols: 7
         date  sessions  unique_visitors  page_views  bounce_rate  \
0  2013-01-01      9760             7253       39093         0.01   
1  2013-01-02     10456             8151       47611         0.00   
2  2013-01-03     10076             7458       36963         0.00   
3  2013-01-04      9973             8063       53078         0.01   
4  2013-01-05     10223             7882       36790         0.01   

   avg_session_duration_sec  traffic_source  
0                    102.90  organic_search  
1                    120.50  organic_search  
2                    263.60          direct  
3                    151.80          direct  
4                    168.60        referral  
Đọc file thành công
   Rows : 3,652
   Cols : 7

--- 3 dòng đầu ---
        date  sessions  unique_visitors  page_views  bounce_rate  avg_session_duration_sec 